## Sprint 3: Dimensionality Reduction & PCA Visualizer

We have 5 sentence embeddings living in 768-dimensional space.
We can't draw 768 axes. PCA finds the 2 directions that preserve
>
the most structure and projects everything down onto them.

Steps:
1. Mean-center the data
2. Compute the covariance matrix
3. Extract eigenvectors with numpy.linalg.eigh
4. Project embeddings onto the top 2 eigenvectors
5. Plot and prove the clusters are real

---

***Setup Cell***

Run this cell first, confirm you see All magnitudes 1.0: True, then run the PCA cells in order. Everything will be defined.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sentences = [
    "Cats are independent and curious animals.",
    "Dogs are loyal and love to play fetch.",
    "A kitten is a young cat that loves to nap.",
    "A microprocessor executes billions of instructions per second.",
    "The CPU is the central processing unit of a computer.",
]

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
raw_embeddings = np.array(model.encode(sentences))

def l2_norm_scratch(v):
    return np.sqrt(np.sum(v ** 2))

def normalize(v):
    return v / l2_norm_scratch(v)

def cosine_similarity(a, b):
    return np.sum(a * b) / (l2_norm_scratch(a) * l2_norm_scratch(b))

embeddings = np.array([normalize(v) for v in raw_embeddings])

print(f"embeddings shape: {embeddings.shape}")
print(f"All magnitudes 1.0: {np.allclose([l2_norm_scratch(v) for v in embeddings], 1.0)}")

---

**Mean-center the data**

PCA requires the data to be centered at the origin first. Why? Because we care
about the *spread* of the data around the mean, 
>
not the absolute position of the whole cloud in space.

If all your sentences happened to have a large positive values in dimension 47, that's not meaningful variance -- it's just an offset. Subtracting the mean removes that offset.


In [ ]:
mean_vector = np.mean(embeddings, axis=0)
X_centered = embeddings - mean_vector

print("Original mean (first 5 dims):", np.round(np.mean(embeddings, axis=0)[:5], 4))
print("Centered mean (first 5 dims):", np.round(np.mean(X_centered, axis=0)[:5], 6))

**Computing Covariance Matrix**

The *covariance matrix* tells us that for every pair of dimensions, do they tend up to go up and down together?
>
If dimension 47 and dimension 312 both tend to be high for animal-related sentences and low for tech sentences, they are correlated. 
>
The covariance matrix captures all of this.

In [ ]:
n = X_centered.shape[0]
cov_matrix = (1 / (n - 1)) * (X_centered.T @ X_centered)

print(f"Covariance matrix shape: {cov_matrix.shape}") 
print(f"It is symmetric: {np.allclose(cov_matrix, cov_matrix.T)}")

**Extracting *eigenvectors***

The function *numpy.linalg.eigh* solves the eigenvalue problem for symmetric matrices. 
>
It returns:
- eigenvalues - how much variance each direction captures (scalar per component)
- eigenvectors - the actual directions (one 768-D vector per component)
>
eigh returns them in ASCENDING order (smallest-biggest), so we reverse to get the most important components first.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Reverse so index 0 = most important
eigenvalues = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]

total_variance = np.sum(eigenvalues)
explained_variance =   eigenvalues / total_variance * 100

print("Top 5 components — variance explained:")
for i in range(5):
    print(f"  PC{i+1}: {explained_variance[i]:.1f}%")

print(f"\nPC1 + PC2 together: {explained_variance[0] + explained_variance[1]:.1f}%") # For sentence embeddings, 30-60% accross 2 components is typical.

**Projecting it down to 2D**

In [ ]:
top_2_components = eigenvectors[:, :2]

embeddings_2d = X_centered @ top_2_components

print("Original shape:", embeddings.shape)         
print("Projected shape:", embeddings_2d.shape)    
print("\n2D coordinates for each sentence:")
for i, label in enumerate(["Cats", "Dogs", "Kitten", "Microprocessor", "CPU"]):
    print(f"  {label:>15}: ({embeddings_2d[i, 0]:.4f}, {embeddings_2d[i, 1]:.4f})")

**Plot to prove the clusters**

In [ ]:
labels = ["Cats", "Dogs", "Kitten", "Microprocessor", "CPU"]
colors = ["#1D9E75", "#1D9E75", "#1D9E75", "#534AB7", "#534AB7"]
# Green for animals, purple for tech

fig, ax = plt.subplots(figsize=(8, 6))

for i, (label, color) in enumerate(zip(labels, colors)):
    ax.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1],
               color=color, s=180, zorder=3)
    ax.annotate(label,
                xy=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
                xytext=(10, 6), textcoords="offset points",
                fontsize=11, color=color, fontweight="bold")

ax.axhline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.4)
ax.axvline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.4)
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)", fontsize=11)
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)", fontsize=11)
ax.set_title("PCA Projection — 768D → 2D\n(implemented from scratch, no Scikit-learn)",
             fontsize=12)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("../images/pca_projection.png", dpi=150, bbox_inches="tight")
plt.show()

**Mathematical Proof**

Compute pairwise Euclidean distances IN THE 2D PROJECTED SPACE and confirm they match the intuition from our cosine similarity matrix.

In [ ]:
def euclidean_2d(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

cats_2d         = embeddings_2d[0]
dogs_2d         = embeddings_2d[1]
micro_2d        = embeddings_2d[3]

dist_cats_dogs  = euclidean_2d(cats_2d, dogs_2d)
dist_cats_micro = euclidean_2d(cats_2d, micro_2d)

print("=== Spatial Proof in 2D Projected Space ===")
print(f"Distance Cats → Dogs:          {dist_cats_dogs:.4f}")
print(f"Distance Cats → Microprocessor:{dist_cats_micro:.4f}")
print()
if dist_cats_dogs < dist_cats_micro:
    print("✓ PROVED: Cats and Dogs are spatially closer than Cats and Microprocessor")
    print(f"  Cats-Dogs is {dist_cats_micro / dist_cats_dogs:.1f}x closer than Cats-Microprocessor")
else:
    print("Unexpected result — check your embeddings")

print()
print("=== Cross-check against Sprint 2 cosine similarity ===")
print(f"Cosine sim Cats↔Dogs:          {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"Cosine sim Cats↔Microprocessor:{cosine_similarity(embeddings[0], embeddings[3]):.4f}")
print("Both methods agree on which pair is more similar ✓")


## Sprint 3: Summary & Key Takeaways

### What We Were Trying to Do
We had 5 sentence embeddings — each a point in 768-dimensional space.
We couldn't draw 768 axes, so we needed a mathematical way to compress
the data down to 2 dimensions without losing the structure that matters.

---

### The Core Idea: Spread = Information
The fundamental insight of PCA is that **variance equals information**.

If all 5 sentences have nearly identical values along some dimension,
that dimension tells you nothing about how the sentences differ from
each other — it's noise. But if values vary wildly across sentences
along some direction, that direction is carrying signal.

PCA finds the directions of *maximum spread* — the axes where your
data varies the most — and uses those as the new compressed dimensions.

$$\text{more spread} \rightarrow \text{higher variance}
\rightarrow \text{more information preserved}$$

---

### The Four Steps

**Step 1 — Mean Centering**
Subtract the mean vector from every embedding so the entire data cloud
is repositioned around the origin. PCA measures spread *relative to
the center*, so centering is required before anything else.

$$\mathbf{X}_{\text{centered}} = \mathbf{X} - \bar{\mathbf{X}}$$

**Step 2 — Covariance Matrix**
Build a $768 \times 768$ grid that measures how much every pair of
dimensions varies together across the 5 sentences. This is the
relationship map PCA uses to find its axes.

$$\mathbf{C} = \frac{1}{n-1} \mathbf{X}_{\text{centered}}^T
\mathbf{X}_{\text{centered}}$$

**Step 3 — Eigenvectors and Eigenvalues**
Decompose the covariance matrix to find its natural axes.

- **Eigenvectors** → the directions of maximum spread (principal components)
- **Eigenvalues** → how much variance each direction captures

$$\mathbf{C}\mathbf{v} = \lambda\mathbf{v}$$

Where $\mathbf{v}$ is an eigenvector and $\lambda$ is its eigenvalue.
The eigenvector with the largest $\lambda$ is PC1 — the single most
informative direction in the data. The second largest is PC2.

**Step 4 — Projection**
Multiply the centered data by the top 2 eigenvectors to compress
every sentence from 768 numbers down to 2 coordinates.

$$\mathbf{X}_{\text{2D}} = \mathbf{X}_{\text{centered}} \cdot
\mathbf{V}_{\text{top2}}$$

$$\underbrace{(5 \times 768)}_{\text{embeddings}}
\cdot \underbrace{(768 \times 2)}_{\text{top 2 eigenvectors}}
= \underbrace{(5 \times 2)}_{\text{2D coordinates}}$$

---

### What the Plot Proved
Plotting the 2D coordinates showed the animal sentences (Cats, Dogs,
Kitten) clustered in one region and the tech sentences (Microprocessor,
CPU) in another — visually confirming the latent space geometry.

PCA did not create those clusters. They existed in 768-dimensional
space all along. PCA just found the best window to see them through.

---

### The Cross-Check
Computing Euclidean distance in the 2D projected space confirmed:

$$d(\text{Cats}, \text{Dogs}) \ll d(\text{Cats}, \text{Microprocessor})$$

And this agreed with the cosine similarity scores from Sprint 2 —
proving the projection preserved the true structure of the data.
